# Proyecto Centinela · Fase 1 · Prueba de acoplamiento y test de procedencia

Este cuaderno produce las evidencias de los anexos A y B de la Propuesta de Datos:

1. Descarga RoCoLe v2 desde Mendeley Data (DOI 10.17632/c5yvn32dzg.2) y calcula su huella SHA-256.
2. Reporta la huella de crudeza: filas × columnas, faltantes y balance de la etiqueta.
3. Revisa los metadatos EXIF de las fotografías: fecha de toma y GPS.
4. Descarga el clima mensual de NASA POWER para el sitio de colecta y calcula su huella.
5. Une diez imágenes reales con la serie por la clave (sitio, ventana) e imprime la tabla del anexo A.

Se ejecuta en Google Colab o en VS Code con «Ejecutar todo».

In [ ]:
import hashlib, json, os, zipfile, datetime, glob, io
import requests
import pandas as pd
import numpy as np
from PIL import Image, ExifTags

FECHA_DESCARGA = datetime.date.today().isoformat()
DATA_DIR = "datos"
os.makedirs(DATA_DIR, exist_ok=True)

# Sitio de colecta publicado por los autores (CIIDEA, ESPAM MFL, Calceta, Manabí, Ecuador)
LAT, LON = -0.834206, -80.177159
SITIO = "CIIDEA-ESPAM MFL"
VENTANA = ("2018-05", "2019-04")   # ventana asumida: 12 meses previos a la publicación (2019-05-17)

def sha256(ruta):
    h = hashlib.sha256()
    with open(ruta, "rb") as f:
        for bloque in iter(lambda: f.read(1 << 20), b""):
            h.update(bloque)
    return h.hexdigest()

print("Fecha de descarga:", FECHA_DESCARGA)

## 1. Descarga de RoCoLe desde la fuente primaria

Ruta A (automática): la API pública de Mendeley Data lista los archivos del DOI y los descarga uno por uno.
Ruta B (manual, si la A falla): abre https://data.mendeley.com/datasets/c5yvn32dzg/2, pulsa «Download All» y guarda el .zip dentro de la carpeta `datos/`. El cuaderno detecta cualquier .zip en esa carpeta.

In [ ]:
DIR_ROCOLE = os.path.join(DATA_DIR, "rocole")
os.makedirs(DIR_ROCOLE, exist_ok=True)
URL_API = "https://data.mendeley.com/public-api/datasets/c5yvn32dzg/files"
procedencia = []   # (archivo, url, fecha, sha256) para el anexo B

zips = glob.glob(os.path.join(DATA_DIR, "*.zip"))
if zips:
    for z_ruta in zips:
        procedencia.append((os.path.basename(z_ruta), "https://doi.org/10.17632/c5yvn32dzg.2", FECHA_DESCARGA, sha256(z_ruta)))
        with zipfile.ZipFile(z_ruta) as z:
            z.extractall(DIR_ROCOLE)
    print("Ruta B: se usó el .zip descargado a mano.")
else:
    r = requests.get(URL_API, params={"folder_id": "root", "version": 2}, timeout=120)
    if r.status_code != 200:
        raise RuntimeError(f"La API respondió {r.status_code}. Usa la ruta B: descarga el .zip a mano en {DATA_DIR}/ y vuelve a ejecutar.")
    archivos = r.json()
    print("Archivos en el DOI:", len(archivos))
    for a in archivos:
        nombre = a["filename"]
        url = a["content_details"]["download_url"]
        destino = os.path.join(DIR_ROCOLE, nombre)
        if not os.path.exists(destino):
            with requests.get(url, stream=True, timeout=600) as d:
                d.raise_for_status()
                with open(destino, "wb") as f:
                    for bloque in d.iter_content(1 << 20):
                        f.write(bloque)
        procedencia.append((nombre, url, FECHA_DESCARGA, sha256(destino)))
    print("Ruta A: descarga desde la API de Mendeley Data.")

# Algunos archivos del DOI son .zip internos: se extraen
for interno in glob.glob(os.path.join(DIR_ROCOLE, "**", "*.zip"), recursive=True):
    with zipfile.ZipFile(interno) as z:
        z.extractall(os.path.splitext(interno)[0])

for nombre, url, fecha, h in procedencia:
    print(f"{nombre} | SHA-256: {h}")

for raiz, dirs, archivos_ in os.walk(DIR_ROCOLE):
    nivel = raiz.replace(DIR_ROCOLE, "").count(os.sep)
    print("  " * nivel + os.path.basename(raiz) + "/", f"({len(archivos_)} archivos)")

## 2. Huella de crudeza (antes de limpiar)

Se buscan los archivos de anotaciones (CSV, XLSX o JSON) y se reportan filas × columnas, faltantes y balance.
Si el cuaderno no identifica la columna de la etiqueta, ajusta `COL_ETIQUETA` con el nombre que muestre `df.columns`.

In [ ]:
anot = [p for p in glob.glob(os.path.join(DIR_ROCOLE, "**", "*"), recursive=True)
        if p.lower().endswith((".csv", ".xlsx", ".xls", ".json"))]
print("Archivos de anotaciones encontrados:")
for p in anot:
    print(" -", os.path.relpath(p, DIR_ROCOLE), "| SHA-256:", sha256(p))

def leer(p):
    if p.lower().endswith(".csv"):
        return pd.read_csv(p)
    if p.lower().endswith((".xlsx", ".xls")):
        return pd.read_excel(p)
    with open(p, encoding="utf-8") as f:
        return pd.json_normalize(json.load(f))

tablas = {}
for p in anot:
    try:
        tablas[p] = leer(p)
    except Exception as e:
        print("No se pudo leer", os.path.basename(p), "->", e)

for p, df in tablas.items():
    print("\n==", os.path.basename(p), "| filas × columnas:", df.shape)
    print("Faltantes (%):", round(df.isna().mean().mean() * 100, 2))
    print("Columnas:", list(df.columns)[:30])
    display(df.head(3))

In [ ]:
# Elige la tabla y la columna de etiqueta (se intenta detectar sola)
COL_ETIQUETA = None
df_anot = None
for p, df in tablas.items():
    for c in df.columns:
        valores = df[c].astype(str).str.lower()
        if valores.str.contains("healthy|rust|red_spider|mite").mean() > 0.5:
            df_anot, COL_ETIQUETA = df, c
            break
    if COL_ETIQUETA:
        break

if COL_ETIQUETA:
    print("Tabla:", os.path.basename(p), "| columna de etiqueta:", COL_ETIQUETA)
    print(df_anot[COL_ETIQUETA].value_counts(dropna=False))
else:
    print("Ajusta COL_ETIQUETA a mano con las columnas impresas arriba.")

## 3. Metadatos EXIF: fecha de toma y GPS

Si las fotografías traen la fecha de toma, la clave puede pasar de agregada (sitio, ventana) a por muestra (sitio, semana).
El GPS se borra antes de subir imágenes a Drive o GitHub (anexo C, bloque F).

In [ ]:
imagenes = sorted(p for p in glob.glob(os.path.join(DIR_ROCOLE, "**", "*"), recursive=True)
                  if p.lower().endswith((".jpg", ".jpeg", ".png")))
print("Imágenes encontradas:", len(imagenes))

TAGS = {v: k for k, v in ExifTags.TAGS.items()}
def exif_info(ruta):
    try:
        ex = Image.open(ruta).getexif()
    except Exception:
        return None, False
    fecha = ex.get(TAGS["DateTime"]) or ex.get_ifd(0x8769).get(TAGS["DateTimeOriginal"]) if ex else None
    gps = bool(ex.get_ifd(0x8825)) if ex else False
    return fecha, gps

muestra = imagenes[:200]
info = [exif_info(p) for p in muestra]
con_fecha = sum(1 for f, _ in info if f)
con_gps = sum(1 for _, g in info if g)
print(f"De {len(muestra)} imágenes revisadas: {con_fecha} con fecha EXIF, {con_gps} con GPS EXIF")
print("Ejemplos de fechas:", [f for f, _ in info if f][:5])

## 4. Serie climática de NASA POWER para el sitio

In [ ]:
URL_POWER = "https://power.larc.nasa.gov/api/temporal/monthly/point"
params = dict(parameters="T2M,RH2M,PRECTOTCORR", community="AG",
              latitude=LAT, longitude=LON, start=2018, end=2019, format="JSON")
r = requests.get(URL_POWER, params=params, timeout=120)
r.raise_for_status()
datos = r.json()["properties"]["parameter"]

clima = (pd.DataFrame(datos)
           .reset_index().rename(columns={"index": "periodo"}))
clima = clima[~clima["periodo"].str.endswith("13")]          # se quita el promedio anual (mes 13)
clima["mes"] = pd.to_datetime(clima["periodo"], format="%Y%m").dt.strftime("%Y-%m")
clima = clima[["mes", "T2M", "RH2M", "PRECTOTCORR"]]

CSV_POWER = os.path.join(DATA_DIR, "nasa_power_ciidea_mensual.csv")
clima.to_csv(CSV_POWER, index=False)
print("Consulta:", r.url)
print("Archivo:", os.path.basename(CSV_POWER), "| SHA-256:", sha256(CSV_POWER))
display(clima)

## 5. Anexo A · Diez filas reales unidas por la clave (sitio, ventana)

Cada imagen toma la media climática de la ventana de colecta declarada.
Que el valor se repita en las diez filas es la salvedad de «rasgo casi constante» que declara la Propuesta.

In [ ]:
en_ventana = clima[(clima["mes"] >= VENTANA[0]) & (clima["mes"] <= VENTANA[1])]
clima_ventana = en_ventana[["T2M", "RH2M", "PRECTOTCORR"]].mean().round(2)

def rgb_medio(ruta):
    arr = np.asarray(Image.open(ruta).convert("RGB").resize((256, 256)), dtype=float)
    return arr.reshape(-1, 3).mean(axis=0).round(1)

rng = np.random.default_rng(42)
elegidas = rng.choice(imagenes, size=10, replace=False)

filas = []
for i, p in enumerate(elegidas, 1):
    r_, g_, b_ = rgb_medio(p)
    filas.append({
        "N.º": i,
        "Clave": f"({SITIO}, {VENTANA[0]} a {VENTANA[1]})",
        "Dato de la rama de imagen": f"{os.path.basename(p)} · RGB medio = ({r_}, {g_}, {b_})",
        "Dato de la serie": f"T2M = {clima_ventana['T2M']} °C · RH2M = {clima_ventana['RH2M']} % · PRECTOTCORR = {clima_ventana['PRECTOTCORR']} mm/día",
    })
anexo_a = pd.DataFrame(filas)
display(anexo_a)
print(anexo_a.to_markdown(index=False))

## 6. Resumen para pegar en los anexos

Copia estas líneas en el anexo B de la Propuesta.

In [ ]:
print("Fecha de descarga:", FECHA_DESCARGA)
for nombre, url, fecha, h in procedencia:
    print(f"| {nombre} | {url} | {fecha} | {h} |")
print(f"| {os.path.basename(CSV_POWER)} | {URL_POWER} | {FECHA_DESCARGA} | {sha256(CSV_POWER)} |")
if df_anot is not None:
    print("Huella de crudeza:", df_anot.shape, "| faltantes (%):", round(df_anot.isna().mean().mean() * 100, 2))
print("Imágenes:", len(imagenes))